# Adani Ports Stock Market Prediction Using Machine Learning
**Developer:** Abhiraj Kumar | B.Tech Computer Science Engineering  
**Model:** XGBoost Regressor  
**Dataset:** ADANIPORTS.csv (Historical Stock Data 2007 - 2021)  
---
## Project Overview
This notebook demonstrates an end-to-end Machine Learning pipeline to analyze historical stock market data of Adani Ports (`ADANIPORTS.csv`) and build an XGBoost Regression model to predict closing stock prices based on data-leakage-free historical features.

### Step 1: Import Required Dependencies

In [ ]:
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')

### Step 2: Data Loading & Initial Inspection

In [ ]:
df = pd.read_csv('../data/ADANIPORTS.csv')
df.columns = [str(c).strip() for c in df.columns]
print('Dataset Shape:', df.shape)
display(df.head())
df.info()

### Step 3: Data Cleaning & Preprocessing

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)
numeric_cols = ['Prev Close', 'Open', 'High', 'Low', 'Last', 'Close', 'VWAP', 'Volume', 'Turnover']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
print('Cleaned Shape:', df.shape)

### Step 4: Exploratory Data Analysis & Visualizations

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df['Date'], df['Close'], label='Closing Price', color='#0052CC')
plt.title('Adani Ports Historical Closing Price (2007-2021)')
plt.xlabel('Date')
plt.ylabel('Price (INR)')
plt.legend()
plt.show()

### Step 5: Data-Leakage-Free Feature Engineering

In [ ]:
data = df.copy()
data['Prev_Close'] = data['Close'].shift(1)
data['Prev_Open'] = data['Open'].shift(1)
data['Prev_High'] = data['High'].shift(1)
data['Prev_Low'] = data['Low'].shift(1)
data['Prev_Volume'] = data['Volume'].shift(1)
data['Daily_Price_Change'] = data['Prev_Close'] - data['Prev_Open']
data['Daily_Pct_Change'] = data['Prev_Close'].pct_change() * 100
data['MA_7'] = data['Prev_Close'].rolling(7).mean()
data['MA_30'] = data['Prev_Close'].rolling(30).mean()
data['Rolling_Std_7'] = data['Prev_Close'].rolling(7).std()
data['Year'] = data['Date'].dt.year
data['Month'] = data['Date'].dt.month
data['Day'] = data['Date'].dt.day
data['Day_of_Week'] = data['Date'].dt.dayofweek
data = data.dropna().reset_index(drop=True)
print('Engineered Dataset Shape:', data.shape)

### Step 6: Chronological Train-Test Split (80/20) & MinMaxScaler Feature Scaling

In [ ]:
feature_cols = ['Prev_Close', 'Prev_Open', 'Prev_High', 'Prev_Low', 'Prev_Volume', 'Daily_Price_Change', 'Daily_Pct_Change', 'MA_7', 'MA_30', 'Rolling_Std_7', 'Year', 'Month', 'Day', 'Day_of_Week']
X = data[feature_cols]
y = data['Close']
split_idx = int(len(data) * 0.80)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f'Train shape: {X_train_scaled.shape}, Test shape: {X_test_scaled.shape}')

### Step 7: Model Training & Multi-Model Evaluation

In [ ]:
model = XGBRegressor(n_estimators=300, learning_rate=0.03, max_depth=5, subsample=0.8, colsample_bytree=0.8, random_state=42)
model.fit(X_train_scaled, y_train)
preds = model.predict(X_test_scaled)
mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)
print(f'XGBoost Regressor Metrics -> MAE: INR {mae:.4f}, RMSE: INR {rmse:.4f}, R2 Score: {r2:.4f}')

### Step 8: Save Model Artifacts for Streamlit Deployment

In [ ]:
joblib.dump(model, '../models/xgboost_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(feature_cols, '../models/feature_columns.pkl')
print('Model artifacts successfully serialized!')